# Script 5 — Análise de Cenários Estratégicos + LLM
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Provedor padrão: **Google Gemini 1.5 Flash** (gratuito via AI Studio).  
Alternativas: Anthropic Claude · OpenAI GPT-4o · Ollama (local).


In [ ]:

import pandas as pd, numpy as np, pickle, warnings, os, json
from pathlib import Path
import joblib

warnings.filterwarnings('ignore')
PASTA_SAIDA = Path('outputs')

# ── Configuração do LLM ────────────────────────────────────────────────────
# Defina a variável de ambiente ou edite aqui:
LLM_PROVEDOR = os.getenv('LLM_PROVEDOR', 'gemini')   # gemini | claude | openai | ollama
GEMINI_API_KEY  = os.getenv('GEMINI_API_KEY',  'SUA_CHAVE_AQUI')
CLAUDE_API_KEY  = os.getenv('ANTHROPIC_API_KEY','SUA_CHAVE_AQUI')
OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY',   'SUA_CHAVE_AQUI')
OLLAMA_MODEL    = os.getenv('OLLAMA_MODEL',      'llama3')

print(f"✅ Configuração: provedor={LLM_PROVEDOR}")


In [ ]:

# Carrega artefatos
dataset  = pd.read_parquet(PASTA_SAIDA / 'dataset_cvm_consolidado.parquet')
with open(PASTA_SAIDA / 'features.pkl','rb') as f: FEATURES = pickle.load(f)
with open(PASTA_SAIDA / 'targets.pkl','rb') as f:  TARGETS  = pickle.load(f)
print(f"Dataset: {dataset.shape} | Features: {len(FEATURES)} | Targets: {TARGETS}")


## Empresas âncora e últimos KPIs disponíveis

In [ ]:

EMPRESAS_ANCORA = {
    'Petrobras':       'Petróleo',
    'Equatorial Energia': 'Energia',
    'Magazine Luiza':  'Varejo',
    'Vale':            'Commodities',
    'WEG':             'Tecnologia',
}

def ultimos_kpis(dataset, empresa):
    """Retorna o vetor de KPIs do último exercício disponível para a empresa."""
    df = dataset[dataset['NOME_CIA'] == empresa].sort_values('ANO')
    if df.empty:
        return None, None
    ultimo = df.iloc[-1]
    kpis_disp = [k for k in FEATURES if k in df.columns and pd.notna(ultimo.get(k))]
    return ultimo[kpis_disp].to_dict(), int(ultimo['ANO'])

for empresa in EMPRESAS_ANCORA:
    kpis, ano = ultimos_kpis(dataset, empresa)
    if kpis:
        print(f"{empresa} — ano base: {ano} | {len(kpis)} KPIs disponíveis")
    else:
        print(f"⚠️  {empresa} não encontrada no dataset")


## Definição dos 3 cenários estratégicos

In [ ]:

CENARIOS = {
    'Expansao_Alavancagem': {
        'descricao': 'Captação de nova dívida LP = 0,5× PL; pressão em cobertura de juros e liquidez',
        'ajustes': {
            'endividamento':     lambda x: x * 1.30,   # +30% endividamento
            'alavancagem_de':    lambda x: x * 1.50,   # +50% D/E
            'liquidez_corrente': lambda x: x * 0.90,   # -10% liquidez
            'cobertura_juros':   lambda x: x * 0.75,   # -25% cobertura
        },
    },
    'Eficiencia_Operacional': {
        'descricao': 'Redução de 10% nos custos operacionais; melhora nas margens',
        'ajustes': {
            'margem_ebit':    lambda x: min(x * 1.15, 0.5),
            'margem_ebitda':  lambda x: min(x * 1.12, 0.6),
            'margem_liquida': lambda x: min(x * 1.10, 0.4),
            'roe':            lambda x: x * 1.08,
        },
    },
    'CAPEX_Capacidade': {
        'descricao': 'Aumento de 30% em CAPEX; pressão de liquidez no curto prazo',
        'ajustes': {
            'giro_ativo':        lambda x: x * 0.92,   # imobilizado cresce mais rápido
            'liquidez_corrente': lambda x: x * 0.85,   # caixa utilizado
            'liquidez_imediata': lambda x: x * 0.80,
            'fco_receita':       lambda x: x * 0.95,
        },
    },
}
print(f"✅ {len(CENARIOS)} cenários configurados")
for nome, c in CENARIOS.items():
    print(f"  {nome}: {c['descricao']}")


## Módulo LLM — adaptador multi-provedor

In [ ]:

def chamar_llm(prompt: str) -> str:
    """
    Chama o LLM configurado e retorna a análise em texto.
    Troca de provedor alterando LLM_PROVEDOR no topo do notebook.
    """
    if LLM_PROVEDOR == 'gemini':
        return _chamar_gemini(prompt)
    elif LLM_PROVEDOR == 'claude':
        return _chamar_claude(prompt)
    elif LLM_PROVEDOR == 'openai':
        return _chamar_openai(prompt)
    elif LLM_PROVEDOR == 'ollama':
        return _chamar_ollama(prompt)
    else:
        return f"[LLM_PROVEDOR '{LLM_PROVEDOR}' não reconhecido]"

def _chamar_gemini(prompt):
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        modelo = genai.GenerativeModel('gemini-1.5-flash')
        resp = modelo.generate_content(prompt)
        return resp.text
    except Exception as e:
        return f"[Erro Gemini: {e}]"

def _chamar_claude(prompt):
    try:
        import anthropic
        client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
        msg = client.messages.create(
            model='claude-sonnet-4-5',
            max_tokens=1024,
            messages=[{'role':'user','content':prompt}]
        )
        return msg.content[0].text
    except Exception as e:
        return f"[Erro Claude: {e}]"

def _chamar_openai(prompt):
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model='gpt-4o',
            messages=[{'role':'user','content':prompt}]
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f"[Erro OpenAI: {e}]"

def _chamar_ollama(prompt):
    try:
        import requests
        r = requests.post('http://localhost:11434/api/generate',
            json={'model':OLLAMA_MODEL,'prompt':prompt,'stream':False})
        return r.json().get('response','')
    except Exception as e:
        return f"[Erro Ollama: {e}]"

print("✅ Módulo LLM configurado")


## Execução dos cenários

In [ ]:

def aplicar_cenario(kpis_base: dict, cenario: dict, features: list) -> np.ndarray:
    """Aplica os ajustes do cenário ao vetor de KPIs e retorna array para predição."""
    kpis = kpis_base.copy()
    for kpi, func in cenario['ajustes'].items():
        if kpi in kpis:
            kpis[kpi] = func(kpis[kpi])
    # Retorna apenas as features usadas pelo modelo
    return np.array([kpis.get(f, 0.0) for f in features]).reshape(1, -1)

def construir_prompt(empresa, setor, ano_base, cenario_nome, cenario_desc,
                     predicoes_base, predicoes_cenario):
    """Monta o prompt para análise executiva pelo LLM."""
    linhas_pred = []
    for target, (val_base, val_cen) in predicoes_base.items():
        variacao = (val_cen - val_base) / abs(val_base) * 100 if val_base != 0 else 0
        linhas_pred.append(
            f"  - {target}: base={val_base:,.0f} → cenário={val_cen:,.0f} "
            f"({variacao:+.1f}%)"
        )
    pred_texto = '\n'.join(linhas_pred)
    return f"""
Você é um analista financeiro sênior especialista em empresas brasileiras de capital aberto.

EMPRESA: {empresa} | SETOR: {setor} | ANO BASE: {ano_base}
CENÁRIO: {cenario_nome}
DESCRIÇÃO: {cenario_desc}

IMPACTO NOS INDICADORES FINANCEIROS (R$ mil):
{pred_texto}

Forneça uma análise executiva em português (máx. 300 palavras) abordando:
1. Avaliação do impacto financeiro projetado para {empresa}
2. Riscos e oportunidades específicos do setor {setor}
3. Recomendação estratégica para gestores

Seja objetivo, use linguagem gerencial e cite os números.
"""

# ── EXECUÇÃO ─────────────────────────────────────────────────────────────────
todos_resultados = []

for empresa, setor in EMPRESAS_ANCORA.items():
    kpis_base, ano_base = ultimos_kpis(dataset, empresa)
    if kpis_base is None:
        print(f"⚠️  {empresa}: sem dados — pulando")
        continue
    print(f"\n{'='*60}")
    print(f"EMPRESA: {empresa} | SETOR: {setor} | Ano base: {ano_base}")
    print('='*60)

    for cenario_nome, cenario in CENARIOS.items():
        print(f"\n  Cenário: {cenario_nome}")
        X_base = np.array([kpis_base.get(f,0.) for f in FEATURES]).reshape(1,-1)
        X_cen  = aplicar_cenario(kpis_base, cenario, FEATURES)

        predicoes_base = {}
        predicoes_cenario = {}
        for target in TARGETS:
            caminho = PASTA_SAIDA / f'melhor_modelo_{target}.pkl'
            if not caminho.exists():
                continue
            modelo = joblib.load(caminho)
            val_base = float(modelo.predict(X_base)[0])
            val_cen  = float(modelo.predict(X_cen)[0])
            predicoes_base[target]     = (val_base, val_cen)
            predicoes_cenario[target]  = val_cen
            variacao = (val_cen-val_base)/abs(val_base)*100 if val_base!=0 else 0
            print(f"    {target}: {val_base:,.0f} → {val_cen:,.0f} ({variacao:+.1f}%)")

        # Chamada ao LLM
        prompt = construir_prompt(
            empresa, setor, ano_base,
            cenario_nome, cenario['descricao'],
            predicoes_base, predicoes_cenario
        )
        analise = chamar_llm(prompt)
        print(f"\n  Análise LLM ({LLM_PROVEDOR}):")
        print(analise[:500] + ('...' if len(analise)>500 else ''))

        todos_resultados.append({
            'empresa': empresa, 'setor': setor, 'ano_base': ano_base,
            'cenario': cenario_nome,
            **{f'{t}_base': predicoes_base[t][0] for t in predicoes_base},
            **{f'{t}_cenario': predicoes_base[t][1] for t in predicoes_base},
            'analise_llm': analise,
        })


In [ ]:

# Salva resultados
df_resultados = pd.DataFrame(todos_resultados)
df_resultados.to_csv(PASTA_SAIDA / 'resultados_cenarios.csv', index=False, encoding='utf-8-sig')
df_resultados.to_excel(PASTA_SAIDA / 'resultados_cenarios.xlsx', index=False)
print(f"\n✅ {len(df_resultados)} análises de cenário salvas em:")
print(f"  {PASTA_SAIDA}/resultados_cenarios.csv")
print(f"  {PASTA_SAIDA}/resultados_cenarios.xlsx")


## Tabela-resumo comparativa entre cenários

In [ ]:

if not df_resultados.empty:
    import warnings; warnings.filterwarnings('ignore')
    # Para cada empresa, mostra os 3 cenários lado a lado
    for empresa in EMPRESAS_ANCORA:
        df_emp = df_resultados[df_resultados['empresa']==empresa]
        if df_emp.empty: continue
        print(f"\n{empresa}")
        cols_show = ['cenario'] + [c for c in df_emp.columns if '_cenario' in c]
        print(df_emp[cols_show].set_index('cenario').T.to_string())
print("\n✅ Pipeline completo concluído!")
